# TextLab: Fine-tune RoBERTa for stress detection (Google Colab)

This notebook fine-tunes `roberta-base` on Dreaddit for **6 experiments**:

| Experiment | Trains on | Tests on |
|---|---|---|
| `standard / all` | all 5 topics of the training split | all 715 test posts |
| `new_topic / <topic>` (x5) | the other 4 topics of the training split | that topic's test posts |

Each run tunes on a **validation split taken from the training data** (20%, stratified, seed 42 — the
same split the TF-IDF baseline uses) with **early stopping** on validation F1. The test posts are never
used for training or model selection.

## Step-by-step

1. **Turn on the GPU:** menu **Runtime → Change runtime type → T4 GPU → Save**.
2. **Upload the data:** click the 📁 **Files** icon on the left and upload these two files from your
   project folder (or wait for step 3 to show an upload button):
   - `data/processed/train.csv`
   - `data/processed/test.csv`
3. **Run everything:** menu **Runtime → Run all**. The progress bars show each experiment
   (6 trainings in total).
4. **Download the results:** the last cell downloads **`textlab_roberta_outputs.zip`**. If it does not
   start, open the 📁 Files panel and download it from there.
5. **On your laptop**, unzip it into the project folder:
   - `roberta/` folder → `models/roberta/` (so you have `models/roberta/config.json`)
   - `roberta.csv` → `results/predictions/roberta.csv`
   - `roberta_training.csv` → `results/roberta_training.csv`

   Then run in PowerShell from the project root:
   ```powershell
   .\.venv\Scripts\Activate.ps1
   python -m scripts.evaluate
   ```
   This adds RoBERTa (with bootstrap confidence intervals) to `results/metrics.csv` and the charts, and the
   API will use RoBERTa automatically the next time it starts.

In [ ]:
# 1. Install / check libraries (Colab already has torch and transformers).
!pip install -q "transformers>=4.44" "accelerate>=0.30" scikit-learn pandas
import torch
print("GPU available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only - enable the GPU (step 1)!")

In [ ]:
# 2. Settings. Keep SEED = 42 so results can be repeated.
MODEL_NAME = "roberta-base"
SEED = 42
MAX_LENGTH = 256          # Dreaddit posts are short; 256 tokens covers almost all of them
LEARNING_RATE = 2e-5
BATCH_SIZE = 16
MAX_EPOCHS = 6
PATIENCE = 2              # early stopping: stop after 2 epochs without validation-F1 gain
VAL_SIZE = 0.2            # same validation split as the TF-IDF baseline
TOPICS = ["abuse", "anxiety", "money", "ptsd", "relationships"]
OUTPUT_DIR = "/content/outputs"

In [ ]:
# 3. Load the processed data (upload it if it is not there yet).
import os
import pandas as pd

if not (os.path.exists("train.csv") and os.path.exists("test.csv")):
    from google.colab import files
    print("Please choose train.csv and test.csv from data/processed/ ...")
    files.upload()

train_df = pd.read_csv("train.csv", dtype={"id": str})
test_df = pd.read_csv("test.csv", dtype={"id": str})
assert len(train_df) == 2838 and len(test_df) == 715, "Unexpected file sizes - wrong files?"
print(f"train: {len(train_df)} posts | test: {len(test_df)} posts")
print(train_df.groupby("topic")["label"].agg(posts="size", stress_rate="mean").round(3))

In [ ]:
# 4. Helper functions: dataset, metrics, training and prediction.
import json
import shutil
from datetime import datetime, timezone

import numpy as np
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          EarlyStoppingCallback, Trainer, TrainingArguments, set_seed)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collator = DataCollatorWithPadding(tokenizer)


class PostDataset(torch.utils.data.Dataset):
    # Tokenized posts (and labels) for the Trainer.
    def __init__(self, df):
        self.encodings = tokenizer(df["text"].tolist(), truncation=True, max_length=MAX_LENGTH)
        self.labels = df["label"].tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.encodings.items()}
        item["labels"] = self.labels[i]
        return item


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return {"f1": f1_score(labels, logits.argmax(-1), pos_label=1)}


def train_model(train_part, run_dir):
    # Fine-tune on train_part; pick the best epoch by F1 on a validation split of train_part.
    set_seed(SEED)
    fit_part, val_part = train_test_split(train_part, test_size=VAL_SIZE,
                                          stratify=train_part["label"], random_state=SEED)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    warmup_steps = int(0.1 * MAX_EPOCHS * np.ceil(len(fit_part) / BATCH_SIZE))   # 10% warmup
    args = TrainingArguments(
        output_dir=run_dir, learning_rate=LEARNING_RATE, num_train_epochs=MAX_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=64,
        weight_decay=0.01, warmup_steps=warmup_steps, eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="f1", greater_is_better=True,
        save_total_limit=1, fp16=torch.cuda.is_available(), seed=SEED, data_seed=SEED,
        logging_strategy="epoch", report_to="none",
    )
    trainer = Trainer(model=model, args=args, train_dataset=PostDataset(fit_part),
                      eval_dataset=PostDataset(val_part), data_collator=collator,
                      compute_metrics=compute_metrics,
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])
    trainer.train()
    history = [log for log in trainer.state.log_history if "eval_f1" in log]
    best = max(history, key=lambda log: log["eval_f1"])
    info = {"best_epoch": best["epoch"], "val_f1": best["eval_f1"], "epochs_run": history[-1]["epoch"],
            "n_fit": len(fit_part), "n_val": len(val_part)}
    return trainer, info


def stress_probabilities(trainer, df):
    logits = trainer.predict(PostDataset(df)).predictions
    return torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1)[:, 1].numpy()


def prediction_frame(setting, test_set, df, probs):
    # Same columns as results/predictions/*.csv in the project.
    return pd.DataFrame({"setting": setting, "test_set": test_set, "id": df["id"].values,
                         "topic": df["topic"].values, "label": df["label"].values,
                         "stress_prob": probs, "pred": (probs > 0.5).astype(int)})

In [ ]:
# 5. Run the 6 experiments: standard split first, then leave-one-topic-out for each topic.
experiments = [("standard", "all", train_df, test_df)] + [
    ("new_topic", t, train_df[train_df["topic"] != t], test_df[test_df["topic"] == t]) for t in TOPICS
]
os.makedirs(OUTPUT_DIR, exist_ok=True)
prediction_frames, training_rows = [], []

for setting, test_set, train_part, test_part in experiments:
    print(f"\n===== {setting} / {test_set}: train {len(train_part)} posts, test {len(test_part)} posts =====")
    run_dir = f"/content/runs/{setting}_{test_set}"
    trainer, info = train_model(train_part.reset_index(drop=True), run_dir)
    probs = stress_probabilities(trainer, test_part.reset_index(drop=True))
    frame = prediction_frame(setting, test_set, test_part, probs)
    prediction_frames.append(frame)
    quick_f1 = f1_score(frame["label"], frame["pred"])
    training_rows.append({"setting": setting, "test_set": test_set, **info, "test_f1": quick_f1})
    print(f"best epoch {info['best_epoch']:.0f} | val F1 {info['val_f1']:.3f} | test F1 {quick_f1:.3f}")

    if setting == "standard":   # keep only the main model for the web app
        model_dir = f"{OUTPUT_DIR}/roberta"
        trainer.save_model(model_dir)
        tokenizer.save_pretrained(model_dir)
        with open(f"{model_dir}/metadata.json", "w") as fh:
            json.dump({"model_name": "roberta", "base_model": MODEL_NAME,
                       "trained_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                       "seed": SEED, "max_length": MAX_LENGTH, "learning_rate": LEARNING_RATE,
                       "batch_size": BATCH_SIZE, **info}, fh, indent=2)
    shutil.rmtree(run_dir, ignore_errors=True)   # free disk space
    del trainer
    torch.cuda.empty_cache()

In [ ]:
# 6. Save predictions and training summary (bootstrap CIs are computed on your laptop by scripts.evaluate).
pd.concat(prediction_frames, ignore_index=True).to_csv(f"{OUTPUT_DIR}/roberta.csv", index=False)
summary = pd.DataFrame(training_rows)
summary.to_csv(f"{OUTPUT_DIR}/roberta_training.csv", index=False)
summary.round(3)

In [ ]:
# 7. Zip everything and download it.
shutil.make_archive("/content/textlab_roberta_outputs", "zip", OUTPUT_DIR)
print("Created /content/textlab_roberta_outputs.zip")
try:
    from google.colab import files
    files.download("/content/textlab_roberta_outputs.zip")
except ImportError:
    pass